In [ ]:
# ================================================================
# SAFARA APD — YOLO11 Training on Kaggle T4 GPU
# Repo   : https://github.com/FauzanFx/training-yolo-safara
# GPU    : NVIDIA Tesla T4 (16 GB VRAM)
# Limit  : 11 jam (batas aman dari 12 jam Kaggle)
# ================================================================

import os, shutil, yaml, torch, subprocess


# ── 0. Konstanta ─────────────────────────────────────────────
REPO_URL    = "https://github.com/FauzanFx/training-yolo-safara"
REPO_DIR    = "/kaggle/working/training-yolo-safara"
OUTPUT_DIR  = "/kaggle/working/results"
RUN_NAME    = "safara_t4"
WEIGHTS_DIR = os.path.join(OUTPUT_DIR, RUN_NAME, "weights")

# Roboflow API key — wajib diset di Kaggle → Add-ons → Secrets
# dengan nama "ROBOFLOW_API_KEY" sebelum menjalankan notebook ini.
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception as e:
    raise RuntimeError(
        "ROBOFLOW_API_KEY tidak ditemukan di Kaggle Secrets. "
        "Buka Add-ons → Secrets → tambahkan key bernama 'ROBOFLOW_API_KEY'."
    ) from e


# ── 1. Install dependencies ──────────────────────────────────
print("[1/5] Installing dependencies...")
subprocess.run(["pip", "install", "-q", "ultralytics", "roboflow"], check=True)


# ── 2. Clone repository ──────────────────────────────────────
print("[2/5] Cloning repository...")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(f"  cwd: {os.getcwd()}")


# ── 3. Download dataset dari Roboflow ────────────────────────
print("[3/5] Downloading dataset from Roboflow...")
from roboflow import Roboflow
rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("yolotraining-gmh0n").project("safara-object-detection")
version = project.version(4)
dataset = version.download("yolov11", location=os.path.join(REPO_DIR, "dataset"))


# ── 4. Patch data.yaml ───────────────────────────────────────
print("[4/5] Patching data.yaml...")
yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

config["nc"]    = 8
config["names"] = [
    "hairnet", "jas_lab", "kacamata_pelindung", "masker",
    "orang",   "sarung_tangan", "tangan", "wajah",
]

with open(yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("  data.yaml patched:", config["names"])


# ── 5. Adaptive LR callback ──────────────────────────────────
def make_adaptive_lr_callback(patience=10, factor=0.5, min_lr=1e-7):
    """
    ReduceLROnPlateau untuk Ultralytics YOLO.
    Memantau training loss tiap epoch; jika tidak membaik selama
    `patience` epoch, LR dikali `factor`. Modifikasi dilakukan
    pada initial_lr optimizer DAN base_lrs scheduler agar cosine
    decay bawaan YOLO tetap konsisten.
    """
    state = {'best_loss': float('inf'), 'wait': 0}

    def on_train_epoch_end(trainer):
        warmup = int(getattr(trainer.args, 'warmup_epochs', 3))
        if trainer.epoch < warmup:
            return

        try:
            tloss = trainer.tloss
            loss  = float(tloss.mean()) if hasattr(tloss, 'mean') else float(tloss)
        except Exception:
            loss  = float(trainer.loss.mean()) if hasattr(trainer.loss, 'mean') else float(trainer.loss)

        if loss < state['best_loss'] - 1e-4:
            state['best_loss'] = loss
            state['wait']      = 0
        else:
            state['wait'] += 1

        if state['wait'] >= patience:
            current_lrs = [pg.get('initial_lr', pg['lr']) for pg in trainer.optimizer.param_groups]
            if max(current_lrs) <= min_lr:
                print(f"[AdaptiveLR] Sudah di min_lr={min_lr:.1e}, tidak dikecilkan lagi.")
                state['wait'] = 0
                return

            for pg in trainer.optimizer.param_groups:
                old              = pg.get('initial_lr', pg['lr'])
                pg['initial_lr'] = max(old * factor, min_lr)
                pg['lr']         = max(pg['lr'] * factor, min_lr)

            if hasattr(trainer, 'scheduler') and hasattr(trainer.scheduler, 'base_lrs'):
                trainer.scheduler.base_lrs = [
                    max(blr * factor, min_lr)
                    for blr in trainer.scheduler.base_lrs
                ]

            new_lr = max(pg['lr'] for pg in trainer.optimizer.param_groups)
            print(
                f"\n[AdaptiveLR] Epoch {trainer.epoch} — loss stuck {patience} epoch "
                f"(best={state['best_loss']:.4f}) → LR dikecilkan menjadi {new_lr:.2e}"
            )
            state['wait'] = 0

    return {'on_train_epoch_end': on_train_epoch_end}


# ── 6. Training ──────────────────────────────────────────────
print("[5/5] Starting YOLO training on T4 GPU...")
print(f"  CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU  : {props.name} ({props.total_memory / 1e9:.1f} GB VRAM)")

from ultralytics import YOLO

model = YOLO(os.path.join(REPO_DIR, "yolo11n.pt"))

lr_callback = make_adaptive_lr_callback(patience=10, factor=0.5, min_lr=1e-7)
model.add_callback('on_train_epoch_end', lr_callback['on_train_epoch_end'])

model.train(
    data          = yaml_path,
    epochs        = 300,     # Batas atas epoch; time=11.0 yang akan menghentikan duluan
    time          = 11.0,    # Hard limit 11 jam — YOLO selesaikan epoch berjalan lalu berhenti
    imgsz         = 640,
    batch         = 32,      # T4 16 GB VRAM → batch 32 aman untuk YOLO11n
    device        = 0,
    workers       = 4,
    cache         = True,    # Cache dataset ke RAM, tiap epoch lebih cepat
    amp           = True,    # Mixed precision FP16 → 2× lebih cepat di T4
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    cos_lr        = True,    # Cosine decay sebagai base schedule
    patience      = 50,      # Early stop jika 50 epoch tanpa improvement
    save_period   = 10,      # Checkpoint tiap 10 epoch
    project       = OUTPUT_DIR,
    name          = RUN_NAME,
    exist_ok      = True,
    pretrained    = True,
    optimizer     = "AdamW",
    mosaic        = 1.0,
    mixup         = 0.1,
    copy_paste    = 0.1,     # Bantu recall — salin objek kecil ke gambar lain
    degrees       = 10.0,
    fliplr        = 0.5,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    verbose       = True,
)


# ── 7. Evaluasi akhir ────────────────────────────────────────
print("\nRunning final validation...")
best_pt = os.path.join(WEIGHTS_DIR, "best.pt")
last_pt = os.path.join(WEIGHTS_DIR, "last.pt")

model_best = YOLO(best_pt)
metrics    = model_best.val(data=yaml_path, batch=16, device=0, workers=4)

print(f"\n{'='*50}")
print(f"  mAP50    : {metrics.box.map50:.4f}")
print(f"  mAP50-95 : {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall   : {metrics.box.mr:.4f}")
print(f"{'='*50}")

print("\nPer-class results:")
class_names = config["names"]
for i, name in enumerate(class_names):
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    ap   = metrics.box.ap[i]   if i < len(metrics.box.ap)   else 0
    print(f"  {name:22s}: mAP50={ap50:.4f}  mAP50-95={ap:.4f}")


# ── 8. Salin best.pt & last.pt ke root /kaggle/working ───────
# File di /kaggle/working otomatis tersedia di tab Output untuk diunduh.
for src, dst in [
    (best_pt, "/kaggle/working/best.pt"),
    (last_pt, "/kaggle/working/last.pt"),
]:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  [Saved] {dst}")
    else:
        print(f"  [Skip]  {src} tidak ditemukan")

print("\nDONE — Unduh best.pt dan last.pt dari tab Output.")